In [1]:
# 02_members_features
#
# 목적: members_v3.csv를 정제하고 파생 피처(tenure_days)를 추가한다.
#      여기서는 "결정적 규칙 기반" 정제만 수행한다 (plan 문서 1단계):
#        - bd(나이) 이상치는 결측(NaN)으로만 마킹하고, 실제 대체값(중앙값)은 채우지 않는다.
#          -> 중앙값은 train split의 분포에서만 계산해야 하므로(리키지 방지),
#             그 처리는 분할 이후 별도 단계(모델링 직전)에서 수행한다.
#
# members_v3.csv 컬럼 설명
# msno                    : 유저 고유 ID
# city                    : 거주 도시 코드 (범주형, 21종)
# bd                      : 나이(age). 0 이하/100 초과는 이상치로 간주해 결측 처리
# gender                  : 성별 ('male'/'female'/결측) -> 결측은 'unknown'으로 명시
# registered_via          : 가입 경로 코드 (범주형, 18종)
# registration_init_time  : 가입일 (YYYYMMDD) -> tenure_days(가입 경과일수) 파생에 사용
#
# 입력: data/raw/members_v3.csv
# 출력: data/processed/features_members.csv

In [2]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 두 코호트(2월/3월 만료) 피처를 하나의 컷오프로 통일하기로 한 결정 (plan 문서 참고)
CUTOFF_DATE = pd.Timestamp("2017-02-28")

BD_MIN_VALID = 1
BD_MAX_VALID = 100

In [3]:
df = pd.read_csv(RAW_DIR / "members_v3.csv")
print(df.shape)
df.head()

(6769473, 6)


,msno,city,bd,gender,registered_via,registration_init_time
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,0,NaN,11,20110911
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,0,NaN,7,20110914
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,0,NaN,11,20110915
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,0,NaN,11,20110915
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32,female,9,20110915


In [4]:
# bd(나이) 이상치 -> 결측 마킹 (고정 임계값이라 분할 전에 해도 안전)
invalid_bd = (df["bd"] < BD_MIN_VALID) | (df["bd"] > BD_MAX_VALID)
print(f"이상치로 결측 처리되는 bd 값: {invalid_bd.sum():,} ({invalid_bd.mean() * 100:.1f}%)")

df["bd_is_missing"] = invalid_bd.astype("int8")
df["bd_clean"] = df["bd"].where(~invalid_bd, other=pd.NA)

이상치로 결측 처리되는 bd 값: 4,545,866 (67.2%)


In [5]:
# gender 결측 -> 'unknown' 명시적 카테고리
df["gender"] = df["gender"].fillna("unknown")
print(df["gender"].value_counts())

gender
unknown    4429505
male       1195355
female     1144613
Name: count, dtype: int64


In [6]:
# 가입 경과일수(tenure_days) 파생
registration_date = pd.to_datetime(df["registration_init_time"], format="%Y%m%d")
tenure_days_raw = (CUTOFF_DATE - registration_date).dt.days

# 가입일이 컷오프 이후인 비정상 케이스(음수 tenure) -> 0으로 클리핑 (고정 규칙)
neg_tenure = (tenure_days_raw < 0).sum()
print(f"음수 tenure_days(가입일이 컷오프 이후) 건수: {neg_tenure:,} -> 0으로 클리핑")

df["tenure_days"] = tenure_days_raw.clip(lower=0)
print(df["tenure_days"].describe())

음수 tenure_days(가입일이 컷오프 이후) 건수: 154,166 -> 0으로 클리핑


count    6.769473e+06
mean     8.076939e+02
std      8.407916e+02
min      0.000000e+00
25%      2.690000e+02
50%      5.070000e+02
75%      1.043000e+03
max      4.722000e+03
Name: tenure_days, dtype: float64


In [7]:
features_members = df[["msno", "city", "registered_via", "bd_clean", "bd_is_missing", "gender", "tenure_days"]]

print(features_members.shape)
print(features_members.isna().sum())
features_members.head()

(6769473, 7)


msno                    0
city                    0
registered_via          0
bd_clean          4545866
bd_is_missing           0
gender                  0
tenure_days             0
dtype: int64


,msno,city,registered_via,bd_clean,bd_is_missing,gender,tenure_days
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,11,NaN,1,unknown,1997
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,7,NaN,1,unknown,1994
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,11,NaN,1,unknown,1993
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,11,NaN,1,unknown,1993
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,9,32.0,0,female,1993


In [8]:
features_members.to_csv(PROCESSED_DIR / "features_members.csv", index=False)
print(f"저장 완료: {PROCESSED_DIR / 'features_members.csv'} ({len(features_members):,} rows)")

저장 완료: ..\data\processed\features_members.csv (6,769,473 rows)
